# 🧑‍⚖️ LLM Output Evaluation

**Day 9 — Notebook 2 of 4 | Estimated Time: 30 minutes**

---

## What You'll Learn
- The four core dimensions for evaluating LLM answers
- Implementing LLM-as-judge with structured Pydantic output
- Faithfulness: detecting hallucinations relative to source context
- Relevance, coherence, and conciseness scoring
- Composite scoring and comparing system variants

---

## The LLM-as-Judge Pattern

Human evaluation is accurate but expensive and slow.  
**LLM-as-judge** uses a capable LLM to score outputs automatically.

```
System output
     │
     ▼
Judge LLM ← (question + context + rubric)
     │
     ▼
Structured score + reasoning
```

Key insight: the judge sees the **rubric explicitly** — this makes its scoring  
more consistent than asking it to "rate this answer."

---

## Setup

In [ ]:
%pip install -U -q "google-genai>=1.0.0"

In [ ]:
import sys, os, json
from statistics import mean

repo_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
if repo_root not in sys.path:
    sys.path.append(repo_root)

from helpers.auth import get_api_key
from google import genai
from google.genai import types
from pydantic import BaseModel
from IPython.display import Markdown

client = genai.Client(api_key=get_api_key())
GEN_MODEL = "gemini-2.5-flash"
print("✅ Ready!")

---

## 1. Test Data: Questions, Context, and Answers

We'll evaluate answers from three hypothetical systems:
- **System A**: Vanilla LLM (no retrieval)
- **System B**: Basic RAG
- **System C**: Advanced RAG with compression

In [ ]:
# Context documents (what System B & C retrieved)
CONTEXTS = {
    "leave-policy": """Leave Policy: Employees receive 25 days of annual leave per year, 
accruing at 2.08 days per month. Leave requests must be submitted via the HR portal at 
least 5 business days in advance. Emergency leave of up to 2 days can be approved 
same-day by a direct manager. Unused leave up to 5 days can be carried over to the 
following year. Leave above 5 days is forfeited on January 1st.""",

    "expense-policy": """Expense Policy: Business expenses must be submitted within 30 days 
of incurrence. Receipts are required for all expenses over £25. Meal expenses are capped 
at £50 per person per day. Hotel accommodation is capped at £200 per night in London 
and £150 per night elsewhere. Expenses are reimbursed within 10 business days of approval.""",

    "remote-work": """Remote Work Policy: Employees may work remotely up to 3 days per week. 
All employees must be in the office on Wednesdays for team collaboration. International 
remote work requires prior approval from HR and is limited to 20 consecutive days. 
A home office allowance of £500 per year is provided to cover equipment costs.""",
}

# Evaluation cases: question, correct context, and three system answers
EVAL_CASES = [
    {
        "id": "case-1",
        "question": "How many days of annual leave do employees get?",
        "context": CONTEXTS["leave-policy"],
        "ground_truth": "25 days per year, accruing at 2.08 days per month.",
        "system_a": "Most companies offer around 20-25 days of annual leave, though this varies "
                    "by country and company policy. In the UK, the statutory minimum is 28 days "
                    "including bank holidays.",
        "system_b": "Employees receive 25 days of annual leave per year.",
        "system_c": "Employees receive 25 days of annual leave per year, accruing at 2.08 days "
                    "per month. Up to 5 unused days can be carried over to the following year.",
    },
    {
        "id": "case-2",
        "question": "What is the hotel expense limit in London?",
        "context": CONTEXTS["expense-policy"],
        "ground_truth": "£200 per night in London.",
        "system_a": "Hotel rates in London are quite high. Companies typically reimburse reasonable "
                    "accommodation costs, usually around £150-200 per night depending on the policy.",
        "system_b": "The hotel accommodation limit is £200 per night in London and £150 elsewhere.",
        "system_c": "Hotel accommodation in London is capped at £200 per night. Outside London, "
                    "the cap is £150 per night.",
    },
    {
        "id": "case-3",
        "question": "Can I work from abroad for a month?",
        "context": CONTEXTS["remote-work"],
        "ground_truth": "International remote work requires HR approval and is limited to 20 consecutive days.",
        "system_a": "Many companies allow occasional remote work from abroad. You should check with "
                    "your HR department about the specific policy and any tax implications.",
        "system_b": "International remote work requires prior approval from HR and is limited to "
                    "20 consecutive days, so a full month would not be permitted.",
        "system_c": "No — international remote work is limited to 20 consecutive days and requires "
                    "prior HR approval. A full calendar month (typically 30+ days) would exceed this limit.",
    },
]

print(f"Evaluation cases: {len(EVAL_CASES)}")
for c in EVAL_CASES:
    print(f"  [{c['id']}] {c['question']}")

---

## 2. Faithfulness — Detecting Hallucinations

In [ ]:
class FaithfulnessScore(BaseModel):
    score: int          # 1–5
    has_hallucination: bool
    hallucinated_claims: list[str]  # Specific claims not supported by context
    reasoning: str


def evaluate_faithfulness(question: str, context: str, answer: str) -> FaithfulnessScore:
    """
    Score how faithfully the answer sticks to the provided context.
    5 = every claim is supported; 1 = major unsupported claims.
    """
    response = client.models.generate_content(
        model=GEN_MODEL,
        contents=f"""Evaluate the FAITHFULNESS of the answer relative to the context.
Faithfulness measures whether all claims in the answer are supported by the context.
Do not reward or penalise based on correctness of the context itself.

Scoring rubric:
5 = All claims fully supported by context. No unsupported additions.
4 = Minor unsupported details that don't change the core answer.
3 = Some claims extrapolated beyond the context.
2 = Multiple important claims not in the context.
1 = Answer is largely not grounded in the provided context.

Context: {context}

Question: {question}

Answer to evaluate: {answer}""",
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=FaithfulnessScore,
            temperature=0.0,
        ),
    )
    return FaithfulnessScore.model_validate_json(response.text)


# Run faithfulness evaluation
print("FAITHFULNESS EVALUATION\n")
print(f"{'Case':<10} {'Sys A':>7} {'Sys B':>7} {'Sys C':>7}")
print("-" * 40)

faith_results = []
for case in EVAL_CASES:
    scores = {}
    for sys_name, answer_key in [("A", "system_a"), ("B", "system_b"), ("C", "system_c")]:
        score = evaluate_faithfulness(
            case["question"], case["context"], case[answer_key]
        )
        scores[sys_name] = score

    faith_results.append(scores)
    halluc_flag = "🚨" if scores["A"].has_hallucination else "  "
    print(
        f"  {case['id']}    {halluc_flag}{scores['A'].score}/5    {scores['B'].score}/5    {scores['C'].score}/5"
    )

# Show hallucinated claims from System A
print("\nHallucinated claims from System A (no context):")
for i, (case, scores) in enumerate(zip(EVAL_CASES, faith_results)):
    if scores["A"].hallucinated_claims:
        print(f"  [{case['id']}] {scores['A'].hallucinated_claims}")

---

## 3. Answer Relevance

In [ ]:
class RelevanceScore(BaseModel):
    score: int       # 1–5
    addresses_question: bool
    missing_aspects: list[str]  # Key aspects of the question the answer missed
    reasoning: str


def evaluate_relevance(question: str, answer: str) -> RelevanceScore:
    """
    Score whether the answer directly addresses what was asked.
    Does NOT require a context — relevance is purely about question-answer alignment.
    """
    response = client.models.generate_content(
        model=GEN_MODEL,
        contents=f"""Evaluate the RELEVANCE of the answer to the question.
Relevance measures whether the answer directly addresses what was asked.

Scoring rubric:
5 = Completely addresses the question, nothing extraneous.
4 = Mostly addresses the question with minor tangents.
3 = Partially addresses the question; missing key aspects.
2 = Loosely related but doesn't answer what was asked.
1 = Does not address the question.

Question: {question}

Answer: {answer}""",
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=RelevanceScore,
            temperature=0.0,
        ),
    )
    return RelevanceScore.model_validate_json(response.text)


print("RELEVANCE EVALUATION\n")
print(f"{'Case':<10} {'Sys A':>7} {'Sys B':>7} {'Sys C':>7}")
print("-" * 40)

rel_results = []
for case in EVAL_CASES:
    scores = {}
    for sys_name, answer_key in [("A", "system_a"), ("B", "system_b"), ("C", "system_c")]:
        score = evaluate_relevance(case["question"], case[answer_key])
        scores[sys_name] = score

    rel_results.append(scores)
    print(f"  {case['id']}     {scores['A'].score}/5    {scores['B'].score}/5    {scores['C'].score}/5")

---

## 4. Correctness — Answer vs Ground Truth

In [ ]:
class CorrectnessScore(BaseModel):
    score: int           # 1–5
    is_correct: bool     # True if the core answer matches ground truth
    errors: list[str]    # Specific factual errors
    reasoning: str


def evaluate_correctness(question: str, answer: str, ground_truth: str) -> CorrectnessScore:
    """Score factual correctness against a known ground truth."""
    response = client.models.generate_content(
        model=GEN_MODEL,
        contents=f"""Evaluate the CORRECTNESS of the answer by comparing it to the ground truth.

Scoring rubric:
5 = Correct and complete — matches ground truth exactly or with equivalent phrasing.
4 = Mostly correct — minor omissions or different phrasing that don't affect accuracy.
3 = Partially correct — gets the main fact right but misses important details.
2 = Mostly incorrect — wrong key facts but related content.
1 = Incorrect — contradicts ground truth or refuses to answer.

Ground truth: {ground_truth}

Question: {question}

Answer: {answer}""",
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=CorrectnessScore,
            temperature=0.0,
        ),
    )
    return CorrectnessScore.model_validate_json(response.text)


print("CORRECTNESS EVALUATION\n")
print(f"{'Case':<10} {'Sys A':>7} {'Sys B':>7} {'Sys C':>7}")
print("-" * 40)

corr_results = []
for case in EVAL_CASES:
    scores = {}
    for sys_name, answer_key in [("A", "system_a"), ("B", "system_b"), ("C", "system_c")]:
        score = evaluate_correctness(
            case["question"], case[answer_key], case["ground_truth"]
        )
        scores[sys_name] = score

    corr_results.append(scores)
    print(f"  {case['id']}     {scores['A'].score}/5    {scores['B'].score}/5    {scores['C'].score}/5")
    for sys_name in ["A"]:
        if scores[sys_name].errors:
            print(f"         Sys {sys_name} errors: {scores[sys_name].errors}")

---

## 5. Composite Score and System Comparison

In [ ]:
def composite_score(
    faithfulness: int,
    relevance: int,
    correctness: int,
    weights: dict | None = None,
) -> float:
    """Weighted composite of individual dimension scores (normalised to 0–1)."""
    if weights is None:
        weights = {"faithfulness": 0.4, "relevance": 0.3, "correctness": 0.3}
    score = (
        weights["faithfulness"] * faithfulness / 5
        + weights["relevance"] * relevance / 5
        + weights["correctness"] * correctness / 5
    )
    return round(score, 3)


print("COMPOSITE SCORE COMPARISON")
print("(Faithfulness 40% | Relevance 30% | Correctness 30%)\n")

print(f"{'Case':<12} {'Sys A':>8} {'Sys B':>8} {'Sys C':>8}")
print("-" * 45)

all_composites = {"A": [], "B": [], "C": []}

for i, case in enumerate(EVAL_CASES):
    row = {}
    for sys_name in ["A", "B", "C"]:
        cs = composite_score(
            faith_results[i][sys_name].score,
            rel_results[i][sys_name].score,
            corr_results[i][sys_name].score,
        )
        row[sys_name] = cs
        all_composites[sys_name].append(cs)

    print(f"  {case['id']}        {row['A']:.3f}    {row['B']:.3f}    {row['C']:.3f}")

avg_a = mean(all_composites["A"])
avg_b = mean(all_composites["B"])
avg_c = mean(all_composites["C"])

print("-" * 45)
print(f"  AVERAGE        {avg_a:.3f}    {avg_b:.3f}    {avg_c:.3f}")

winner = max([("A", avg_a), ("B", avg_b), ("C", avg_c)], key=lambda x: x[1])
print(f"\n  Best system: {winner[0]} (avg composite = {winner[1]:.3f})")

---

## 6. Prompt Sensitivity — Testing Judge Consistency

LLM judges can be sensitive to prompt phrasing. Here's how to measure this.

In [ ]:
# Test same answer with two judge prompt variations
class SimpleScore(BaseModel):
    score: int
    reasoning: str

test_case = EVAL_CASES[0]
test_answer = test_case["system_b"]

PROMPT_V1 = f"""Rate the quality of this answer from 1-5 (5 = excellent).
Context: {test_case['context']}
Question: {test_case['question']}
Answer: {test_answer}"""

PROMPT_V2 = f"""You are a strict quality evaluator. Score from 1-5, where 1=poor and 5=perfect.
Only give 5 if the answer is completely correct and includes all important details.
Context: {test_case['context']}
Question: {test_case['question']}
Answer: {test_answer}"""

# Run each prompt 3 times to check variance
print("Judge consistency test (3 runs each prompt):\n")

for prompt_name, prompt in [("Prompt V1 (lenient)", PROMPT_V1), ("Prompt V2 (strict)", PROMPT_V2)]:
    scores = []
    for run in range(3):
        response = client.models.generate_content(
            model=GEN_MODEL,
            contents=prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=SimpleScore,
                temperature=0.0,
            ),
        )
        result = SimpleScore.model_validate_json(response.text)
        scores.append(result.score)

    variance = max(scores) - min(scores)
    print(f"  {prompt_name}: scores={scores}, variance={variance}")

print("""
💡 Key insight: the same answer can get different scores from different prompts.
   Mitigation strategies:
   • Always use temperature=0.0 for judges
   • Use explicit, numbered rubrics (not vague "1=poor 5=excellent")
   • Run 3 judge calls and take the median/majority vote
   • Validate your judge against human labels on a calibration set
""")

---

## 🏋️ Exercise 1: Add a Conciseness Dimension

Verbose answers waste tokens and can obscure the key information.

In [ ]:
# Exercise 1: Conciseness evaluator
# TODO:
# Write evaluate_conciseness(question, answer) → SimpleScore
# Scoring rubric:
#   5 = Answer is the minimum length needed — no padding or repetition
#   4 = Slightly verbose but all content adds value
#   3 = Noticeable padding or repetition
#   2 = Significantly overlong; much irrelevant content
#   1 = Padded to the point of obscuring the answer
#
# Then run it on all 3 systems for all 3 cases.
# Hypothesis: System A (no retrieval) may be more verbose than B and C.

class ConcisennessScore(BaseModel):
    score: int
    word_count: int
    reasoning: str

def evaluate_conciseness(question: str, answer: str) -> ConcisennessScore:
    # TODO: Implement
    pass


# TODO: Run on all cases and print results

---

## 🏋️ Exercise 2: Calibrate the Judge

Validate your judge against manually assigned scores.

In [ ]:
# Exercise 2: Judge calibration
# The following 5 answers have been manually rated by a human expert.
# Your task: run evaluate_faithfulness() on each, then measure correlation.

CALIBRATION_SET = [
    {
        "context": CONTEXTS["leave-policy"],
        "question": "How many days of leave can I carry over?",
        "answer": "You can carry over up to 5 unused days to the following year.",
        "human_faithfulness_score": 5,
    },
    {
        "context": CONTEXTS["leave-policy"],
        "question": "How many days of leave can I carry over?",
        "answer": "Unused leave can typically be carried over, usually up to 10 days depending on policy.",
        "human_faithfulness_score": 2,
    },
    {
        "context": CONTEXTS["expense-policy"],
        "question": "When must I submit expenses?",
        "answer": "Business expenses must be submitted within 30 days of incurrence.",
        "human_faithfulness_score": 5,
    },
    {
        "context": CONTEXTS["expense-policy"],
        "question": "Do I need a receipt for a £20 lunch?",
        "answer": "No, receipts are required for expenses over £25, so a £20 lunch does not require one.",
        "human_faithfulness_score": 5,
    },
    {
        "context": CONTEXTS["remote-work"],
        "question": "How many remote days per week are allowed?",
        "answer": "Employees can work remotely up to 4 days per week.",
        "human_faithfulness_score": 1,
    },
]

# TODO:
# 1. Run evaluate_faithfulness() on each calibration case
# 2. Print: human score | LLM score | delta | question
# 3. Calculate Mean Absolute Error (MAE) between human and LLM scores
# 4. If MAE > 1.0, identify what made the judge disagree with the human

print("Calibration results:")
print(f"{'Human':>7} {'LLM':>5} {'Delta':>6}  Question")
print("-" * 60)
# TODO: fill in

---

## Key Takeaways

| Dimension | Measures | Requires Context? |
|-----------|----------|-----------------|
| Faithfulness | Are all claims supported by the context? | Yes |
| Relevance | Does the answer address the question? | No |
| Correctness | Does the answer match the ground truth? | No (needs GT) |
| Conciseness | Is the answer appropriately brief? | No |

**Judge design principles:**
- Always use `temperature=0.0` for reproducibility
- Use explicit numbered rubrics in the prompt
- Calibrate against human labels before trusting the judge
- Consider majority vote across 3 judge calls for high-stakes evals

---

## 📚 Further Reading

| Resource | Type | Description |
|----------|------|-------------|
| [G-Eval Paper](https://arxiv.org/abs/2303.16634) | Paper | NLG evaluation using GPT-4 as judge |
| [MT-Bench](https://arxiv.org/abs/2306.05685) | Paper | LLM-judged multi-turn evaluation framework |
| [LangChain Evaluation](https://python.langchain.com/docs/guides/evaluation) | Docs | LangChain's built-in evaluators |

---

**Next up:** [03_RAG_Retrieval_Metrics.ipynb](./03_RAG_Retrieval_Metrics.ipynb)